# Phase 2: Model Training with LightGBM's Built-in Balancer

This notebook demonstrates how to train a `lightgbm.LGBMClassifier` on our imbalanced dataset. We will leverage the `is_unbalance=True` parameter, which is a simple and powerful way to handle class imbalance directly within the model.

**Goal:** Evaluate the initial performance of a balanced LightGBM model.

**Steps:**
1. Load the pre-processed dataset.
2. Split the data into training and testing sets.
3. Initialize and train the `LGBMClassifier` with `is_unbalance=True`.
4. Evaluate the model using a classification report and a confusion matrix.

### 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

print("Libraries imported successfully.")

### 2. Load the Dataset

We load the `final_training_dataset.csv` file created in Phase 1. This file contains our 12 engineered features and the corresponding target labels.

In [ ]:
# CSV written by AIH_preprocessor/stage3.ipynb
DATASET_PATH = os.environ.get('DATASET_PATH', '../AIH_preprocessor/final_training_dataset.csv')

if not os.path.exists(DATASET_PATH):
    print(f"❌ Error: Dataset file not found at '{DATASET_PATH}'")
    print("Please make sure the CSV file from Phase 1 is in the same directory as this notebook.")
else:
    df = pd.read_csv(DATASET_PATH)
    print("✅ Dataset loaded successfully.")
    print(f"Dataset shape: {df.shape}")
    df.head()

### 3. Separate Features (X) and Labels (y)

We split our DataFrame into the features matrix `X` and the target vector `y`.

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

# Define the label mapping for later visualization
label_map = {
    0: "BODY_TEXT", 1: "TITLE", 2: "H1",
    3: "H2", 4: "H3", 5: "H4"
}
label_names = [label_map[i] for i in sorted(y.unique())]

print("Features (X) and labels (y) have been separated.")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

### 4. Split Data into Training and Testing Sets

We'll use a standard 80/20 split. `stratify=y` is crucial here—it ensures that the proportion of each class is the same in both the training and testing sets, which is very important for imbalanced datasets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Data split into training and testing sets.")
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

### 5. Train the LightGBM Model with `is_unbalance=True`

This is the key step. We initialize the `LGBMClassifier` and set `is_unbalance=True`. This tells LightGBM to automatically calculate and apply class weights during training to counteract the imbalance.

In [ ]:
# Initialize the model
lgbm = lgb.LGBMClassifier(
    is_unbalance=True, 
    random_state=42
)

# Train the model
print("Training the LightGBM model with is_unbalance=True...")
lgbm.fit(X_train, y_train)
print("✅ Model training complete.")

### 6. Evaluate Model Performance

Now we use the trained model to make predictions on our held-out test set and evaluate its performance. We will look at:
- **Accuracy**: Overall correctness.
- **Classification Report**: A detailed breakdown of precision, recall, and F1-score for each class.
- **Confusion Matrix**: A visual grid showing what the model predicted versus the actual labels.

In [ ]:
# Make predictions
y_pred = lgbm.predict(X_test)

# --- Accuracy Score ---
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.4f}\n")

# --- Classification Report ---
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

# --- Confusion Matrix ---
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix for LightGBM with is_unbalance=True')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

### 7. Analysis of Results

When you analyze the results, pay close attention to the **recall** and **F1-score** for the minority classes (like `TITLE`, `H3`, `H4`).

- **Recall** tells you: "Of all the actual `TITLE` spans, what percentage did the model correctly identify?" This is crucial because we want to avoid missing these important headings.
- The **F1-score** provides a balanced measure between precision and recall.

The confusion matrix gives a clear visual of any common misclassifications (e.g., if the model frequently confuses `H2` with `H3`). Based on these results, we can decide if this performance is sufficient or if we need to proceed to more advanced techniques like SMOTE.